# Stage 1: LLM-Based Risk Extraction
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Luwam Major Kefali**

**Hilina Fissha Woreta**

This notebook builds Stage 1 of the pipeline: an open-weight LLM reads each patient's discharge summary and extracts structured risk information (comorbidity burden, social determinants of health, psychiatric complexity, discharge risk indicators) as JSON. This structured output feeds into Stage 2 (XGBoost readmission classifier) alongside the tabular features.

Everything here reads from `notes_index.parquet` and `mimic_features.parquet`, produced by the preprocessing notebook, and from `config.yaml` via the shared `config.py` loader, so nothing is hardcoded twice between our all the tracks.

## Setup

Installing the packages we need on top of what Kaggle ships by default. `bitsandbytes` lets us load the model in 4-bit, so that a 7B model comfortably fits on a single Kaggle GPU.

In [2]:
!pip install -q -U transformers accelerate bitsandbytes pydantic pyyaml

In [3]:
import os
import sys
import re
import json

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pydantic import BaseModel, Field, ValidationError
from typing import List, Literal

# Kaggle's config dataset
config_dir = "/kaggle/input/datasets/luwammajor/clinical-pipeline-config"
if config_dir not in sys.path:
    sys.path.append(config_dir)

from config import load_config

# Explicitly pass the path to bypass default lookups
cfg = load_config(config_path="/kaggle/input/datasets/luwammajor/clinical-pipeline-config/config.yaml")

print("active environment:", cfg.active_environment)
print("model:", cfg.stage1.model_name)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

active environment: kaggle
model: BioMistral/BioMistral-7B
GPU available: True
device: Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## Loading the notes

`notes_index.parquet` links each `hadm_id` to the ID of its latest discharge note, but not the note text itself. It was already filtered in preprocessing to only the admissions that survived the complete-labs cohort. We also pull `race_clean` and `readmitted_30d` from `mimic_features.parquet`, we need both in a moment for stratified sampling, before we touch any raw note text.

In [4]:
notes_index = pd.read_parquet(cfg.paths.notes_index)
print("admissions with a discharge note available:", len(notes_index))

strat_cols = pd.read_parquet(
    cfg.paths.features, columns=["hadm_id"] + cfg.stage1.stratify_on
)

notes_index = notes_index.merge(strat_cols, on="hadm_id", how="left")
notes_index = notes_index.dropna(subset=cfg.stage1.stratify_on).reset_index(drop=True)
print("admissions with stratification labels available:", len(notes_index))

admissions with a discharge note available: 293760
admissions with stratification labels available: 293760


## Sampling the cohort for Stage 1

Running a 7B model over the full cohort is not realistic on Kaggle's GPU quota. Rough budget: at roughly 3 to 5 seconds per note (4-bit inference, up to 512 generated tokens), a cohort in the hundreds of thousands of admissions would take somewhere in the hundreds of GPU hours, against a weekly quota of about 30 hours and a total project budget of 100 hours each.

Instead we draw a stratified sample, stratified on `race_clean` and `readmitted_30d` by default (same convention Hilina used for the train/val/test split), so the sample stays representative on both the outcome and the protected attribute we care about most. `stage1.sample_size` in the config controls how many admissions actually get extracted, currently 4,000. At roughly 4 seconds a note that is well under 5 GPU hours, comfortably inside a single Kaggle session.

Rare strata (a race and outcome combination with fewer than 2 admissions) are kept in full rather than dropped, matching the handling in preprocessing.

This is an estimate, not a guarantee, the sanity check further down measures actual generation time on our hardware before we commit to the full run. If it comes in slower, lower `stage1.sample_size` rather than push through a run that will not finish.

In [5]:
from sklearn.model_selection import train_test_split


def stratified_sample(df, strat_cols, n_samples, random_state):
    """Draw a stratified sample of size n_samples, keeping rare strata in full
    rather than dropping them (a stratum needs at least 2 members for
    sklearn's stratified split to work)."""
    strat_key = df[strat_cols].astype(str).agg("_".join, axis=1)
    strat_counts = strat_key.value_counts()

    valid = strat_counts[strat_counts >= 2].index
    rare = strat_counts[strat_counts < 2].index

    main = df[strat_key.isin(valid)].copy()
    leftover = df[strat_key.isin(rare)].copy()

    if n_samples >= len(main):
        sampled_main = main
    else:
        sampled_main, _ = train_test_split(
            main,
            train_size=n_samples,
            stratify=strat_key[strat_key.isin(valid)],
            random_state=random_state,
        )

    return pd.concat([sampled_main, leftover], ignore_index=True)


sampled_index = stratified_sample(
    notes_index,
    strat_cols=cfg.stage1.stratify_on,
    n_samples=cfg.stage1.sample_size,
    random_state=cfg.random_seed,
)

print("Stage 1 sample size:", len(sampled_index))
print("\nrace distribution, full cohort (%):")
print((notes_index["race_clean"].value_counts(normalize=True) * 100).round(1))
print("\nrace distribution, Stage 1 sample (%):")
print((sampled_index["race_clean"].value_counts(normalize=True) * 100).round(1))
print("\nreadmission rate, full cohort:  ", round(notes_index["readmitted_30d"].mean() * 100, 1), "%")
print("readmission rate, Stage 1 sample:", round(sampled_index["readmitted_30d"].mean() * 100, 1), "%")

Stage 1 sample size: 4000

race distribution, full cohort (%):
race_clean
White                     70.0
Black/African American    14.5
Other/Unknown              7.4
Hispanic/Latino            5.0
Asian                      3.1
Name: proportion, dtype: float64

race distribution, Stage 1 sample (%):
race_clean
White                     70.0
Black/African American    14.5
Other/Unknown              7.4
Hispanic/Latino            5.0
Asian                      3.1
Name: proportion, dtype: float64

readmission rate, full cohort:   21.7 %
readmission rate, Stage 1 sample: 21.6 %


Now join the sampled admissions against the raw note text. We filter to the sample before this join rather than after, `discharge.csv` holds every discharge note in MIMIC-IV-Note, there is no reason to carry the other roughly 99 percent of it in memory through the rest of the notebook.

In [6]:
discharge = pd.read_csv(
    f"{cfg.paths.notes_dir}/discharge.csv",
    usecols=["note_id", "text"],
)

notes = sampled_index.merge(discharge, on="note_id", how="left")

missing_text = notes["text"].isna().sum()
print("sampled notes missing raw text after join:", missing_text)

notes = notes.dropna(subset=["text"]).reset_index(drop=True)
print("notes with text available:", len(notes))

sampled notes missing raw text after join: 0
notes with text available: 4000


## Section extraction

Discharge summaries average around 2,200 tokens, and most of that is not useful for risk extraction: sign-off boilerplate, medication reconciliation tables, deidentification placeholders. Feeding the whole note to the model wastes context and slows generation.

MIMIC discharge summaries follow a fairly consistent structure, with section headers as their own line ending in a colon (`Social History:`, `Brief Hospital Course:`, etc). We split on that pattern and keep only the sections we actually want, defined in `config.yaml` under `stage1.sections`.

In [7]:
SECTION_HEADER_PATTERN = re.compile(r"^\s*([A-Z][A-Za-z /\-]{2,40}):\s*$", re.MULTILINE)


def split_into_sections(note_text: str) -> dict:
    """Split a discharge summary into {section_name: section_text}."""
    headers = list(SECTION_HEADER_PATTERN.finditer(note_text))
    sections = {}
    for i, match in enumerate(headers):
        name = match.group(1).strip()
        start = match.end()
        end = headers[i + 1].start() if i + 1 < len(headers) else len(note_text)
        sections[name] = note_text[start:end].strip()
    return sections


def extract_relevant_sections(note_text: str, wanted_sections: list) -> str:
    """Pull just the configured sections, concatenated, for LLM input."""
    sections = split_into_sections(note_text)
    lower_map = {k.lower(): k for k in sections}
    parts = []
    for wanted in wanted_sections:
        key = lower_map.get(wanted.lower())
        if key and sections[key]:
            parts.append(f"{wanted}:\n{sections[key]}")
    return "\n\n".join(parts)


# quick look at what headers a real note actually contains
print(list(split_into_sections(notes.iloc[0]["text"]).keys()))

['Allergies', 'Chief Complaint', 'Major Surgical or Invasive Procedure', 'History of Present Illness', 'Past Medical History', 'Social History', 'Family History', 'Physical Exam', 'Pertinent Results', 'Brief Hospital Course', 'Discharge Medications', 'Discharge Disposition', 'Discharge Diagnosis', 'Discharge Condition', 'Discharge Instructions', 'Followup Instructions']


In [8]:
notes["sections_extracted"] = notes["text"].apply(
    lambda t: extract_relevant_sections(t, cfg.stage1.sections)[: cfg.stage1.max_section_chars]
)

empty_extractions = (notes["sections_extracted"].str.len() == 0).sum()
print(f"notes where none of the target sections were found: {empty_extractions} / {len(notes)}")

# if the regex found nothing, we cannot build a meaningful prompt for that
# note, drop it here and report the count rather than sending an empty
# prompt to the model
notes["sections_extracted"] = notes["sections_extracted"].replace("", np.nan)
notes = notes.dropna(subset=["sections_extracted"]).reset_index(drop=True)
print("notes remaining after dropping empty extractions:", len(notes))

notes where none of the target sections were found: 14 / 4000
notes remaining after dropping empty extractions: 3986


## Loading the model

Using BioMistral-7B by default since it is ungated on Hugging Face and needs no extra setup on Kaggle. `meta-llama/Llama-3.1-8B-Instruct` is a reasonable alternative if we want to compare structured-output reliability later, it just needs an `HF_TOKEN` Kaggle secret and license acceptance first. Swapping models can be done by swapping the model definition line in `config.yaml`.

Loading in 4-bit via `bitsandbytes` so this comfortably fits a single Kaggle T4/P100.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.stage1.load_in_4bit,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(cfg.stage1.model_name)
model = AutoModelForCausalLM.from_pretrained(
    cfg.stage1.model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print("model loaded:", cfg.stage1.model_name)
print("device:", model.device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

### How much does section extraction actually save us

Sanity check on 50 random notes, comparing token count for the full note versus just the extracted sections.

In [ ]:
sample_full = notes["text"].sample(50, random_state=cfg.random_seed)
sample_extracted = notes["sections_extracted"].sample(50, random_state=cfg.random_seed)

full_tok_lens = sample_full.apply(lambda t: len(tokenizer.encode(t)))
extracted_tok_lens = sample_extracted.apply(lambda t: len(tokenizer.encode(t)))

print("avg tokens, full discharge summary:  ", round(full_tok_lens.mean(), 1))
print("avg tokens, extracted sections only: ", round(extracted_tok_lens.mean(), 1))
print("reduction:", round((1 - extracted_tok_lens.mean() / full_tok_lens.mean()) * 100, 1), "%")

## Structured output schema

The model has to return exactly these four fields, nothing else. `social_determinants_flags` uses a fixed vocabulary so we can explode it into binary columns later, the same pattern Hilina used for `cm_*` comorbidity flags in preprocessing. `discharge_risk_indicators` is deliberately free text since risk factors are too varied to force into a fixed category list, we keep the raw phrases for the report and the visualizer, and use the count as a numeric feature for Stage 2.

Using `pydantic` to validate the model's JSON actually matches this shape before we trust it.

In [ ]:
SDOH_CATEGORIES = [
    "housing_instability",
    "food_insecurity",
    "substance_use",
    "limited_social_support",
    "unemployment",
    "transportation_barrier",
]


class RiskExtraction(BaseModel):
    comorbidity_burden_score: int = Field(ge=0, le=10)
    social_determinants_flags: List[str] = Field(default_factory=list)
    psychiatric_complexity: Literal["low", "medium", "high"]
    discharge_risk_indicators: List[str] = Field(default_factory=list)


def extract_json_block(text: str) -> str:
    """Pull the first {...} object out of raw model output, stripping any code fences."""
    text = re.sub(r"```(?:json)?", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("no JSON object found in model output")
    return match.group(0)


def parse_extraction(raw_text: str):
    """Parse and validate raw model output. Returns a dict, or None if it fails."""
    try:
        json_str = extract_json_block(raw_text)
        data = json.loads(json_str)
        validated = RiskExtraction(**data)
        return validated.model_dump() if hasattr(validated, "model_dump") else validated.dict()
    except (ValueError, json.JSONDecodeError, ValidationError):
        return None

## Prompt template

One-shot prompting: a worked example noticeably improves how reliably the model sticks to valid JSON, compared to a bare schema description. Generation is deterministic (`do_sample: false` in the config), so the same note always produces the same extraction, matching the reproducibility commitment in our report.

In [ ]:
SYSTEM_INSTRUCTIONS = f"""You are a clinical information extraction assistant. Given a portion of a hospital discharge summary, extract structured risk information. Respond with ONLY a single JSON object, no explanation, no markdown formatting, matching exactly this schema:

{{
  "comorbidity_burden_score": <integer 0-10, overall severity of comorbidities documented>,
  "social_determinants_flags": [<subset of: {SDOH_CATEGORIES}>],
  "psychiatric_complexity": <"low", "medium", or "high">,
  "discharge_risk_indicators": [<short free-text phrases describing factors that could contribute to readmission risk>]
}}
"""

ONE_SHOT_EXAMPLE_INPUT = """History of Present Illness:
Patient is a 68 year old male with history of CHF and diabetes presenting with shortness of breath.

Social History:
Lives alone, reports difficulty affording medications, no reliable transportation to follow up appointments.

Brief Hospital Course:
Treated for CHF exacerbation, diuresed successfully, glucose control improved.

Discharge Diagnosis:
Acute on chronic CHF exacerbation, type 2 diabetes."""

ONE_SHOT_EXAMPLE_OUTPUT = """{
  "comorbidity_burden_score": 6,
  "social_determinants_flags": ["limited_social_support", "transportation_barrier"],
  "psychiatric_complexity": "low",
  "discharge_risk_indicators": ["lives alone", "medication affordability concerns", "CHF exacerbation history"]
}"""


def build_prompt(note_sections: str) -> str:
    return (
        f"{SYSTEM_INSTRUCTIONS}\n\n"
        f"Example input:\n{ONE_SHOT_EXAMPLE_INPUT}\n\n"
        f"Example output:\n{ONE_SHOT_EXAMPLE_OUTPUT}\n\n"
        f"Now extract from this input:\n{note_sections}\n\n"
        f"Output:"
    )

## Generation

`do_sample=False` gives greedy, deterministic decoding, so `temperature` is only passed when sampling is actually on. Truncating input to 4096 tokens as a hard safety cap, section extraction should already keep us well under that.

In [ ]:
def generate_extraction(prompt: str, tokenizer, model, cfg) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=cfg.stage1.max_new_tokens,
        do_sample=cfg.stage1.do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    if cfg.stage1.do_sample:
        gen_kwargs["temperature"] = cfg.stage1.temperature

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Sanity check on a handful of notes

Before running this on the full sample, look at five real outputs by eye, and time them. This is the point to catch prompt problems or a bad runtime estimate, not after committing several GPU hours to a batch run.

In [ ]:
import time

sample_notes = notes.sample(5, random_state=cfg.random_seed)
timings = []

for row in sample_notes.itertuples():
    prompt = build_prompt(row.sections_extracted)

    start = time.time()
    raw = generate_extraction(prompt, tokenizer, model, cfg)
    elapsed = time.time() - start
    timings.append(elapsed)

    parsed = parse_extraction(raw)

    print(f"hadm_id: {row.hadm_id}  (generation time: {elapsed:.1f}s)")
    print("raw model output:")
    print(raw[:500])
    print("parsed:", parsed)
    print("-" * 80)

avg_time = sum(timings) / len(timings)
projected_hours = (avg_time * len(notes)) / 3600
print(f"\naverage generation time: {avg_time:.1f}s per note")
print(f"projected time for the full sample of {len(notes)} notes: {projected_hours:.1f} GPU hours")
if projected_hours > 8:
    print("this exceeds a single Kaggle session, consider lowering stage1.sample_size in config.yaml")

We check both the outputs and the projected runtime above before continuing. If `parsed` is `None` for most of them, the prompt or schema needs adjusting, not the batch loop below. Common failure modes: the model adds explanatory text before the JSON, or invents a field name that does not match `RiskExtraction`. The `extract_json_block` regex handles the first case, `pydantic` validation catches the second and reports it as a parse failure rather than silently accepting bad data.

If the projected runtime is too long, lower `stage1.sample_size` in the config now, before the batch run below, rather than stopping it partway through.

## Full batch run

Checkpointing every `stage1.checkpoint_every` notes, so a Kaggle session timeout does not lose everything, and the loop resumes automatically from the last checkpoint on rerun.

In [ ]:
def explode_flags(record: dict) -> dict:
    row = {
        "comorbidity_burden_score": record["comorbidity_burden_score"],
        "psychiatric_complexity": record["psychiatric_complexity"],
        "discharge_risk_indicator_count": len(record["discharge_risk_indicators"]),
        "discharge_risk_indicators_text": "; ".join(record["discharge_risk_indicators"]),
    }
    flags = set(record["social_determinants_flags"])
    for cat in SDOH_CATEGORIES:
        row[f"sdoh_{cat}"] = int(cat in flags)
    return row


def run_stage1_batch(notes_df, tokenizer, model, cfg, checkpoint_path):
    results = []
    processed_ids = set()

    if os.path.exists(checkpoint_path):
        existing = pd.read_parquet(checkpoint_path)
        results = existing.to_dict("records")
        processed_ids = set(existing["hadm_id"])
        print(f"resuming from checkpoint: {len(processed_ids)} already processed")

    remaining = notes_df[~notes_df["hadm_id"].isin(processed_ids)]
    print(f"notes remaining to process: {len(remaining)}")

    for i, row in enumerate(remaining.itertuples(), start=1):
        prompt = build_prompt(row.sections_extracted)
        raw_output = generate_extraction(prompt, tokenizer, model, cfg)
        parsed = parse_extraction(raw_output)

        record = {"hadm_id": row.hadm_id, "extraction_success": parsed is not None}
        if parsed is not None:
            record.update(explode_flags(parsed))
        else:
            # neutral defaults so downstream merges do not break.
            # extraction_success=False lets Stage 2 treat these rows
            # separately (e.g. as a missingness indicator) rather than
            # silently mixing failed extractions in with real ones
            record.update({
                "comorbidity_burden_score": np.nan,
                "psychiatric_complexity": "unknown",
                "discharge_risk_indicator_count": 0,
                "discharge_risk_indicators_text": "",
                **{f"sdoh_{c}": 0 for c in SDOH_CATEGORIES},
            })

        results.append(record)

        if i % cfg.stage1.checkpoint_every == 0:
            pd.DataFrame(results).to_parquet(checkpoint_path, index=False)
            print(f"  checkpoint saved at {i} / {len(remaining)} ({len(results)} total so far)")

    final_df = pd.DataFrame(results)
    final_df.to_parquet(checkpoint_path, index=False)
    return final_df

In [ ]:
checkpoint_path = os.path.join(cfg.paths.output_dir, "stage1_checkpoint.parquet")
stage1_raw = run_stage1_batch(notes, tokenizer, model, cfg, checkpoint_path)

print("\nextraction success rate:", round(stage1_raw["extraction_success"].mean() * 100, 1), "%")
print(stage1_raw["extraction_success"].value_counts())

## Saving the output

This is what Stage 2 will merge in on `hadm_id`, alongside the tabular features from preprocessing.

In [ ]:
stage1_raw.to_parquet(cfg.paths.stage1_output, index=False)

print("saved:", cfg.paths.stage1_output)
print("rows:", len(stage1_raw))
print(stage1_raw.head())

## Manual evaluation sample

We don't have ground truth for any of this yet. Our report commits to validating Stage 1 against manually annotated notes, so this samples a set of extractions and writes a CSV with blank columns for us to fill in by hand. This is what the evaluation metric in the report (precision and recall of extracted fields against manual annotations) will actually be computed from.

In [ ]:
eval_sample = notes.merge(stage1_raw, on="hadm_id").sample(
    cfg.stage1.eval_sample_size, random_state=cfg.random_seed
)[
    ["hadm_id", "sections_extracted", "comorbidity_burden_score",
     "psychiatric_complexity", "discharge_risk_indicators_text"]
    + [f"sdoh_{c}" for c in SDOH_CATEGORIES]
].copy()

eval_sample["manual_comorbidity_score"] = ""
eval_sample["manual_psychiatric_complexity"] = ""
eval_sample["manual_notes"] = ""

eval_sample.to_csv(cfg.paths.stage1_eval_sample, index=False)

print(f"saved {len(eval_sample)} rows to {cfg.paths.stage1_eval_sample} for manual annotation")